## Generate DimDate

Range is derived from FactOnlineSales, rounded to full years (time intelligence
breaks on partial years).

Writing by path instead of saveAsTable: the metastore lowercases table names,
a path write keeps the casing.

In [1]:
LAKEHOUSE = "LH_Contoso"

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 5, Finished, Available, Finished, False)

In [2]:
# Generate a date dimension in the lakehouse, sized to the actual fact data.
# Full years: Jan 1 of the first year through Dec 31 of the last year,
# so time intelligence (YTD, PY) works correctly.
# Note: we write by path instead of saveAsTable — the metastore lowercases
# table names, a path write preserves the casing (DimDate).
from pyspark.sql import functions as F

# Derive the date range from FactOnlineSales
bounds = (spark.table(f"{LAKEHOUSE}.dbo.FactOnlineSales")
          .agg(F.min("DateKey").alias("min_date"),
               F.max("DateKey").alias("max_date"))
          .collect()[0])

start_date = f"{bounds['min_date'].year}-01-01"
end_date = f"{bounds['max_date'].year}-12-31"
print(f"Fact range:    {bounds['min_date']} → {bounds['max_date']}")
print(f"DimDate range: {start_date} → {end_date}")

df_date = spark.sql(f"""
    SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS Date
""").select(
    F.col("Date").cast("timestamp").alias("DateKey"),
    F.year("Date").alias("Year"),
    F.quarter("Date").alias("Quarter"),
    F.month("Date").alias("MonthNumber"),
    F.date_format("Date", "MMMM").alias("Month"),
    F.dayofmonth("Date").alias("Day"),
    F.date_format("Date", "EEEE").alias("Weekday"),
)

# Clean up a possibly existing lowercase version from an earlier saveAsTable run
spark.sql(f"DROP TABLE IF EXISTS {LAKEHOUSE}.dbo.dimdate")

# Path write preserves casing; the lakehouse auto-registers the folder as a table
df_date.write.mode("overwrite").format("delta").save("Tables/dbo/DimDate")
print(f"DimDate: {df_date.count()} rows written")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 6, Finished, Available, Finished, False)

Fact range:    2007-01-01 00:00:00 → 2009-12-31 00:00:00
DimDate range: 2007-01-01 → 2009-12-31
DimDate: 1096 rows written
